<a href="https://colab.research.google.com/github/arjr2872003-dot/EV-Energy-Consumption-Analysis/blob/main/employee_handbook_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install pypdf fastembed qdrant-client reportlab pandas


In [13]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

def create_handbook():
    c = canvas.Canvas("employee_handbook.pdf", pagesize=letter)

    # الصفحة الأولى: ساعات العمل والإجازات السنوية
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, 750, "Company Employee Handbook - Part 1")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 700, "1. Working Hours")
    c.setFont("Helvetica", 10)
    c.drawString(50, 680, "Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM.")
    c.drawString(50, 665, "Regular attendance and punctuality are expected from all team members.")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 610, "2. Annual Leave")
    c.setFont("Helvetica", 10)
    c.drawString(50, 590, "Employees receive 18 working days of paid annual leave per year.")
    c.drawString(50, 575, "Requests must be submitted at least seven days in advance and approved by the manager.")
    c.showPage()

    # الصفحة الثانية: الإجازات المرضية والعمل عن بعد
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, 750, "Company Employee Handbook - Part 2")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 700, "3. Sick Leave")
    c.setFont("Helvetica", 10)
    c.drawString(50, 680, "Employees must notify their manager before the workday begins.")
    c.drawString(50, 665, "A medical certificate is required for absences of more than two consecutive days.")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 610, "4. Remote Work Policy")
    c.setFont("Helvetica", 10)
    c.drawString(50, 590, "Employees may work remotely up to two days per week with prior manager approval.")
    c.drawString(50, 575, "Staff members must ensure stable internet connectivity during work hours.")
    c.showPage()

    # الصفحة الثالثة: التدريب والدعم الفني
    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, 750, "Company Employee Handbook - Part 3")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 700, "5. Professional Training")
    c.setFont("Helvetica", 10)
    c.drawString(50, 680, "Employees may request one job-related training course every six months,")
    c.drawString(50, 665, "subject to manager approval and budget availability.")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(50, 610, "6. Technical Support")
    c.setFont("Helvetica", 10)
    c.drawString(50, 590, "Employees should report technical issues to it@company.example and")
    c.drawString(50, 575, "include a short description of the problem along with any relevant error screenshots.")
    c.showPage()

    c.save()
    print("تم إنشاء ملف employee_handbook.pdf بنجاح!")

create_handbook()

تم إنشاء ملف employee_handbook.pdf بنجاح!


In [14]:
from pypdf import PdfReader

# 1. تحديد مسار الملف وقراءته
pdf_path = "employee_handbook.pdf"
reader = PdfReader(pdf_path)

# قائمة لتخزين النصوص المستخرجة مع رقم كل صفحة
extracted_pages = []

# 2. المرور على صفحات الملف صفحة بصفحة
for idx, page in enumerate(reader.pages):
    text = page.extract_text()
    # التحقق من أن الصفحة تحتوي على نص وليست فارغة
    if text and text.strip():
        extracted_pages.append({
            "page_number": idx + 1,
            "text": text.strip()
        })

# 3. عرض عدد الصفحات التي تحتوي على نص
print(f"Total pages with text: {len(extracted_pages)}")

# 4. دمج النصوص وطباعة أول 300 حرف للتأكد من القراءة
combined_text = "\n".join([p["text"] for p in extracted_pages])
print("\n--- First 300 characters of extracted text ---")
print(combined_text[:300])

Total pages with text: 3

--- First 300 characters of extracted text ---
Company Employee Handbook - Part 1
1. Working Hours
Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM.
Regular attendance and punctuality are expected from all team members.
2. Annual Leave
Employees receive 18 working days of paid annual leave per year.
Requests must be submitted at l


In [15]:
# Task 2: Text Chunking

def chunk_text(pages, max_words=120, overlap_words=25, source_file="employee_handbook.pdf"):
    chunks = []
    for p in pages:
        words = p["text"].split()
        chunk_idx = 0
        start = 0

        while start < len(words):
            end = min(start + max_words, len(words))
            chunk_words = words[start:end]
            chunk_text_str = " ".join(chunk_words)

            chunks.append({
                "source": source_file,
                "page": p["page_number"],
                "chunk": chunk_idx,
                "text": chunk_text_str
            })
            chunk_idx += 1
            if end == len(words):
                break
            start += (max_words - overlap_words)

    return chunks

# تطبيق التقطيع بالإعدادات المحددة
chunks = chunk_text(extracted_pages, max_words=120, overlap_words=25)

# طباعة العدد الكلي للـ Chunks
print(f"Total chunks created: {len(chunks)}")

# عرض أول Chunkين
print("\n--- First 2 Chunks ---")
for i in range(min(2, len(chunks))):
    print(f"\nChunk {i+1}:")
    print(f"Source: {chunks[i]['source']}, Page: {chunks[i]['page']}, Chunk ID: {chunks[i]['chunk']}")
    print(f"Text: {chunks[i]['text']}")

Total chunks created: 3

--- First 2 Chunks ---

Chunk 1:
Source: employee_handbook.pdf, Page: 1, Chunk ID: 0
Text: Company Employee Handbook - Part 1 1. Working Hours Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM. Regular attendance and punctuality are expected from all team members. 2. Annual Leave Employees receive 18 working days of paid annual leave per year. Requests must be submitted at least seven days in advance and approved by the manager.

Chunk 2:
Source: employee_handbook.pdf, Page: 2, Chunk ID: 0
Text: Company Employee Handbook - Part 2 3. Sick Leave Employees must notify their manager before the workday begins. A medical certificate is required for absences of more than two consecutive days. 4. Remote Work Policy Employees may work remotely up to two days per week with prior manager approval. Staff members must ensure stable internet connectivity during work hours.


In [16]:
COLLECTION_NAME = "employee_handbook_demo"

In [17]:
# Task 3: Embeddings & Vector Storage

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. تحميل نموذج FastEmbed لإنشاء التضمينات (Embeddings)
embed_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. استخراج النصوص من المقاطع وتوليد المتجهات لها
texts = [c["text"] for c in chunks]
embeddings = list(embed_model.embed(texts))
vector_size = len(embeddings[0])

# 3. إعداد اتصال Qdrant (نستخدم وضع الذاكرة المحلية الخفيف المناسب لـ Colab)
USE_LOCALHOST_QDRANT = False

if USE_LOCALHOST_QDRANT:
    qdrant = QdrantClient("http://localhost:6333")
else:
    qdrant = QdrantClient(":memory:")

# اسم الـ Collection المطلوب في التكليف
COLLECTION_NAME = "employee_handbook_demo"

# حذف المجموعة إذا كانت موجودة مسبقاً ثم إنشاؤها بإعدادات مناسبة للمتجهات
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
)

# 4. تجهيز النقاط (Points) وتخزين المتجهات مع الـ Metadata
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload=chunks[i]  # يشمل source, page, chunk, text
    )
    for i in range(len(chunks))
]

qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

# 5. طباعة عدد المتجهات المخزنة للتأكد
count = qdrant.count(COLLECTION_NAME).count
print(f"Total vectors stored in collection '{COLLECTION_NAME}': {count}")

Total vectors stored in collection 'employee_handbook_demo': 3


In [18]:
# Task 4 & Task 5: Semantic Search & Display Results

# 1. دالة البحث الدلالي في Qdrant
def search_handbook(query, top_k=3):
    # تحويل السؤال إلى متجهات أرقام بنفس نموذج التضمين
    query_vector = list(embed_model.embed([query]))[0].tolist()

    # البحث عن أقرب المتجهات في المجموعة
    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k
    ).points
    return results

# 2. الدالة المطلوبة لعرض وتنسيق النتائج كما في التكليف
def display_results(question, results):
    print("=" * 60)
    print(f"Question: {question}")
    print("=" * 60)
    for idx, res in enumerate(results, start=1):
        score = res.score
        payload = res.payload
        print(f"Result {idx}")
        print(f"Score: {score:.4f}")
        print(f"Source: {payload['source']}")
        print(f"Page: {payload['page']}")
        print(f"Text: {payload['text']}")
        print("-" * 40)
    print("\n")

# 3. قائمة الأسئلة الخمسة المطلوبة للاختبار
test_questions = [
    "What are the company's working hours?",
    "How many annual leave days do employees get?",
    "When is a medical certificate required?",
    "Can employees work remotely?",
    "How can I report a technical problem?"
]

# 4. تنفيذ البحث وعرض النتائج لكل سؤال
search_outputs = {}
for q in test_questions:
    res = search_handbook(q, top_k=3)
    search_outputs[q] = res
    display_results(q, res)

Question: What are the company's working hours?
Result 1
Score: 0.7533
Source: employee_handbook.pdf
Page: 1
Text: Company Employee Handbook - Part 1 1. Working Hours Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM. Regular attendance and punctuality are expected from all team members. 2. Annual Leave Employees receive 18 working days of paid annual leave per year. Requests must be submitted at least seven days in advance and approved by the manager.
----------------------------------------
Result 2
Score: 0.6105
Source: employee_handbook.pdf
Page: 2
Text: Company Employee Handbook - Part 2 3. Sick Leave Employees must notify their manager before the workday begins. A medical certificate is required for absences of more than two consecutive days. 4. Remote Work Policy Employees may work remotely up to two days per week with prior manager approval. Staff members must ensure stable internet connectivity during work hours.
----------------------------------------
Result 3


In [19]:
# Task 6: Prompt Construction

# 1. اختيار أحد الأسئلة وتحديد المقاطع المسترجعة له
sample_question = "Can employees work remotely?"
top_results = search_handbook(sample_question, top_k=3)

# 2. تجميع السياق (Context) مع أرقام الصفحات
context_passages = []
for res in top_results:
    page_num = res.payload['page']
    passage_text = res.payload['text']
    context_passages.append(f"[Page {page_num}]: {passage_text}")

formatted_context = "\n\n".join(context_passages)

# 3. بناء هيكل الـ Prompt بالصيغة المطلوبة بالضبط
rag_prompt = f"""Answer the question using only the context below.
Include the page number supporting your answer.
If the context does not contain the answer, say:
"This information is not available in the provided context."

Context:
{formatted_context}

Question:
{sample_question}
"""

# 4. طباعة الـ Prompt النهائي
print("=" * 60)
print("FINAL GENERATED RAG PROMPT")
print("=" * 60)
print(rag_prompt)

FINAL GENERATED RAG PROMPT
Answer the question using only the context below.
Include the page number supporting your answer.
If the context does not contain the answer, say:
"This information is not available in the provided context."

Context:
[Page 2]: Company Employee Handbook - Part 2 3. Sick Leave Employees must notify their manager before the workday begins. A medical certificate is required for absences of more than two consecutive days. 4. Remote Work Policy Employees may work remotely up to two days per week with prior manager approval. Staff members must ensure stable internet connectivity during work hours.

[Page 1]: Company Employee Handbook - Part 1 1. Working Hours Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM. Regular attendance and punctuality are expected from all team members. 2. Annual Leave Employees receive 18 working days of paid annual leave per year. Requests must be submitted at least seven days in advance and approved by the manager.

[Page 

In [20]:
# 1. اختبار السؤال غير الموجود في الدليل
out_of_scope_query = "Does the company provide free meals?"
out_results = search_handbook(out_of_scope_query, top_k=3)

print("--- نتيجة اختبار سؤال خارج محتوى الدليل ---")
display_results(out_of_scope_query, out_results)

# 2. إنشاء جدول تقييم الأسئلة الخمسة (Evaluation Table)
import pandas as pd

evaluation_data = {
    "Question": [
        "What are the company's working hours?",
        "How many annual leave days do employees get?",
        "When is a medical certificate required?",
        "Can employees work remotely?",
        "How can I report a technical problem?"
    ],
    "Topic": [
        "Working Hours",
        "Annual Leave",
        "Medical Certificate",
        "Remote Work",
        "Technical Support"
    ],
    "Page": [1, 1, 2, 2, 3],
    "Answer Found in Top 3?": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
}

eval_df = pd.DataFrame(evaluation_data)
eval_df

--- نتيجة اختبار سؤال خارج محتوى الدليل ---
Question: Does the company provide free meals?
Result 1
Score: 0.5264
Source: employee_handbook.pdf
Page: 1
Text: Company Employee Handbook - Part 1 1. Working Hours Employees work Sunday through Thursday, from 9:00 AM to 5:00 PM. Regular attendance and punctuality are expected from all team members. 2. Annual Leave Employees receive 18 working days of paid annual leave per year. Requests must be submitted at least seven days in advance and approved by the manager.
----------------------------------------
Result 2
Score: 0.5037
Source: employee_handbook.pdf
Page: 3
Text: Company Employee Handbook - Part 3 5. Professional Training Employees may request one job-related training course every six months, subject to manager approval and budget availability. 6. Technical Support Employees should report technical issues to it@company.example and include a short description of the problem along with any relevant error screenshots.
-------------------

,Question,Topic,Page,Answer Found in Top 3?
0,What are the company's working hours?,Working Hours,1,Yes
1,How many annual leave days do employees get?,Annual Leave,1,Yes
2,When is a medical certificate required?,Medical Certificate,2,Yes
3,Can employees work remotely?,Remote Work,2,Yes
4,How can I report a technical problem?,Technical Support,3,Yes


In [21]:
display(eval_df)


,Question,Topic,Page,Answer Found in Top 3?
0,What are the company's working hours?,Working Hours,1,Yes
1,How many annual leave days do employees get?,Annual Leave,1,Yes
2,When is a medical certificate required?,Medical Certificate,2,Yes
3,Can employees work remotely?,Remote Work,2,Yes
4,How can I report a technical problem?,Technical Support,3,Yes


In [22]:
!pip install gradio -q

In [23]:
import gradio as gr

# دالة تستقبل السؤال من واجهة المستخدم وتسترجع الإجابة
def rag_chat_interface(user_question):
    if not user_question.strip():
        return "الرجاء كتابة سؤال أولاً."

    # 1. البحث الدلالي عن أفضل 3 نتائج
    results = search_handbook(user_question, top_k=3)

    # 2. تنسيق النتائج المسترجعة
    response = f"### 🔍 نتائج البحث عن: **{user_question}**\n\n"

    for idx, res in enumerate(results, start=1):
        score = res.score
        page = res.payload['page']
        text = res.payload['text']

        response += f"**النتيجة {idx}** (درجة التشابه: `{score:.4f}`) — 📄 **صفحة {page}**:\n"
        response += f"> {text}\n\n"
        response += "---\n"

    return response

# بناء الواجهة التفاعلية باستخدام Gradio
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 المساعد الذكي لدليل الموظفين (RAG Assistant)")
    gr.Markdown("اكتب سؤالك للبحث داخل دليل الموظفين واسترجاع النصوص ذات الصلة مع رقم الصفحة مباشرة.")

    with gr.Row():
        with gr.Column():
            question_input = gr.Textbox(
                label="سؤالك (بالإنجليزية):",
                placeholder="e.g. Can employees work remotely?",
                lines=2
            )
            submit_btn = gr.Button("بحث واسترجاع", variant="primary")

            # أمثلة جاهزة يمكن للمستخدم الضغط عليها وتجربتها فوراً
            gr.Examples(
                examples=[
                    ["What are the company's working hours?"],
                    ["How many annual leave days do employees get?"],
                    ["When is a medical certificate required?"],
                    ["Can employees work remotely?"],
                    ["Does the company provide free meals?"]
                ],
                inputs=question_input
            )

        with gr.Column():
            output_display = gr.Markdown(label="النتائج المسترجعة:")

    # ربط الزر بتنفيذ الدالة
    submit_btn.click(
        fn=rag_chat_interface,
        inputs=question_input,
        outputs=output_display
    )

# إطلاق الواجهة داخل Google Colab
demo.launch(share=False)

/tmp/ipykernel_2596/2422457865.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>